# Worked Example: Esk Post Office (040075)

This notebook demonstrates the end-to-end workflow for generating a 100+ year hourly rainfall sequence, following the methodology presented in the HWRS 2025 papers:
*   **Millard et al. (2025)**: *"Of fragments and cubes, a century of continuous rainfall in the Darling Downs"*
*   **Batchelor et al. (2025)**: *"Scaling of historical sub-daily rainfall datasets to represent future climate conditions"*

## Workflow Overview
1.  **Fetch Observed Data**: Get daily rainfall for the ESK gauge via the SILO API.
2.  **Disaggregation**: Convert daily totals to hourly values using the Regionalised Method of Fragments.
3.  **IFD Reconcilation**: Condition the hourly series to match BoM 2016 Intensity-Frequency-Duration (IFD) statistics.
4.  **Climate Change Scaling**: Project the sequence to a 2090 SSP3 horizon using ARR v4.2 uplift and GCM seasonal scaling.

### 1. Fetch Observed Data from SILO
We will fetch historical daily records for **Station 040075 (Esk Post Office)**.

In [ ]:
from pyraingen.silo import get_silo_point_data, prepare_silo_for_pyraingen
import pandas as pd

# Station ID for Esk
site = "040075"
email = "your.email@example.com" # Required for SILO API

# Fetch 40 years of data (1980 - 2020)
silo_df = get_silo_point_data(site, "19800101", "20201231", email)
print(f"Fetched {len(silo_df)} days of daily data.")

# Prepare for disaggregation
daily_rain_array = prepare_silo_for_pyraingen(silo_df, 1980, 2020)

### 2. Sub-daily Disaggregation (Method of Fragments)
We use the `regionalisedsubdailysim` function with `genSeqOption=5` to inject our observed daily data directly.

In [ ]:
from pyraingen.regionalisedsubdailysim import regionalisedsubdailysim
import os

# Configure paths (ensure you have the pluviograph data directory)
path_subdaily = "src/pyraingen/data/example/subdaily/" 
target_idx = 40075

# Run the disaggregation to create an hourly NetCDF
regionalisedsubdailysim(
    fnameInput=None,             # Bypassed by suppliedDailyRain
    pathSubDaily=path_subdaily,
    targetIndex=target_idx,
    suppliedDailyRain=daily_rain_array,
    genSeqOption=5,              # User-supplied data option
    nSims=1,
    fnameSubDaily="esk_hourly_raw.nc"
)

### 3. IFD Conditioning (Reconcile with BoM 2016)
We now ensure the stochastically disaggregated series matches the known IFD characteristics for Esk. `ifdcond` uses an iterative approach to scale the series.

In [ ]:
from pyraingen.ifdcond import ifdcond

# Define target IFDs (typically from BoM Design Rainfall Data System)
target_ifd_csv = "src/pyraingen/data/example/ifd/targetifds.csv"
durations = [60, 360, 720, 1440] # minutes
aeps = [63.2, 50, 20, 10, 5, 2, 1]

ifdcond(
    "esk_hourly_raw.nc", 
    "esk_hourly_conditioned.nc", 
    target_ifd_csv,
    nSims=1,
    TargetIFDdurationsEst=durations,
    TargetIFDdurations=durations,
    AEP=aeps,
    plot=True
)

### 4. Climate Change Scaling (ARR v4.2 & CMIP6)
Following **Batchelor et al. (2025)**, we apply a two-phase scaling to represent a 2090 climate horizon.

In [ ]:
from pyraingen.climate import apply_arr_v4_2_uplift, calculate_submaximal_scaling_factor
import numpy as np

# --- Phase 1: IFD Uplift (Intense Events) ---
delta_temp = 3.5 # Degrees Celsius for 2090 SSP3-7.0
alpha = 5.0      # % increase per degree (ARR v4.2 recommendation)

# Load historical IFD from our conditioned series
historical_ifd_depths = np.array([[32.5, 36.8, 50.9], [42.8, 48.1, 65.1]]) # Simplified example

future_ifd_depths = apply_arr_v4_2_uplift(historical_ifd_depths, alpha, delta_temp)

print("Projected Future IFD Depths (mm):")
print(future_ifd_depths)

# --- Phase 2: Seasonal Volume Adjustment (Sub-maximal) ---
# GCMs predict a 7% reduction in total rainfall for this region
gcm_alpha = 0.93 

vo = 45000.0 # Total observed volume (mm)
vm = 12000.0 # Volume attributed to annual maxima (after uplift)
vnm = 33000.0 # Volume of sub-maximal rainfall

beta = calculate_submaximal_scaling_factor(vo, vm, vnm, gcm_alpha)
print(f"Scaling factor for non-maximal rainfall (beta): {beta:.4f}")

print("\nWorkflow Complete: You now have a future-climate ready continuous rainfall sequence!")